# KvForge Coding Assistant
ProLAD on Qwen2.5-Coder-1.5B-Instruct with CodeAlpaca


In [ ]:
import json, math, time, gc, os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

print("=" * 70)
print("KvForge Coding Assistant — ProLAD on Qwen2.5-Coder-1.5B")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7 else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device} | Dtype: {dtype}")

# ── 1. Model ─────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
print(f"\nLoading {MODEL_ID}...")
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"  {n_params/1e6:.1f}M params")

# ── 2. LoRA Injection ─────────────────────────────────────────────
class LoRALinear(nn.Module):
    def __init__(self, orig, r=8, alpha=16):
        super().__init__()
        self.orig = orig
        self.scaling = alpha / r
        dt = orig.weight.dtype
        self.lora_A = nn.Parameter(torch.randn(orig.in_features, r, dtype=dt) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, orig.out_features, dtype=dt))
        self.active = True
    def activate(self, a=True):
        self.active = a
        self.scaling = (self.orig.lora_alpha / self.orig.r) if a else 0.0
    def forward(self, x):
        h = self.orig.orig(x) if hasattr(self.orig, 'orig') else self.orig(x)
        if self.active:
            h = h + (x @ self.lora_A @ self.lora_B) * (self.orig.lora_alpha / self.orig.r if hasattr(self.orig, 'lora_alpha') else 16/8)
        return h

def inject_lora(model, r=8, alpha=16):
    count = 0
    for n, m in model.named_modules():
        if any(n.endswith(s) for s in [".q_proj", ".k_proj", ".v_proj", ".o_proj"]):
            if isinstance(m, nn.Linear) and not isinstance(m, LoRALinear):
                parent = model; parts = n.split(".")
                for p in parts[:-1]:
                    if p: parent = getattr(parent, p)
                lora_mod = LoRALinear(m, r=r)
                lora_mod.orig.lora_alpha = alpha
                lora_mod.orig.r = r
                setattr(parent, parts[-1], lora_mod)
                count += 1
    return count

def set_lora(model, active):
    for mod in model.modules():
        if hasattr(mod, "activate"):
            mod.activate(active)

def get_loras(model):
    return [(n, m) for n, m in model.named_modules() if hasattr(m, "lora_A")]

n_lora = inject_lora(model, r=8, alpha=16)
lora_p = sum(p.numel() for n, p in model.named_parameters() if "lora" in n)
print(f"LoRA: {n_lora} modules, {lora_p/1e3:.1f}K params")

# ── 3. Dataset — CodeAlpaca ───────────────────────────────────────
print("\nLoading CodeAlpaca...")
from datasets import load_dataset
try:
    ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
except:
    # Fallback: synthetic code prompts
    ds = None

if ds is not None:
    def fmt(x):
        return f"### Instruction:\n{x['instruction']}\n\n### Response:\n{x['output']}"
    texts = [fmt(x) for x in ds if len(x.get('output', '')) > 50][:300]
    print(f"  {len(texts)} samples loaded from CodeAlpaca")
else:
    # Generate synthetic code examples
    print("  Using synthetic code data")
    SYNTHETIC = [
        "### Instruction:\nWrite a Python function to check if a number is prime.\n\n### Response:\ndef is_prime(n):\n    if n < 2: return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0: return False\n    return True",
        "### Instruction:\nWrite a function to compute the nth Fibonacci number.\n\n### Response:\ndef fibonacci(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a",
        "### Instruction:\nImplement a binary search function.\n\n### Response:\ndef binary_search(arr, x):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == x: return mid\n        elif arr[mid] < x: lo = mid + 1\n        else: hi = mid - 1\n    return -1",
        "### Instruction:\nCreate a Python stack class.\n\n### Response:\nclass Stack:\n    def __init__(self): self.items = []\n    def push(self, x): self.items.append(x)\n    def pop(self): return self.items.pop() if self.items else None\n    def peek(self): return self.items[-1] if self.items else None\n    def is_empty(self): return len(self.items) == 0",
    ]
    texts = SYNTHETIC * 25
    print(f"  {len(texts)} synthetic samples")

train_text = "\n\n".join(texts)

# Tokenize and chunk
tokens = tokenizer(train_text, return_tensors="pt", truncation=False)["input_ids"][0]
chunks = [tokens[i:i+512] for i in range(0, len(tokens)-256, 256)][:150]
print(f"  {len(chunks)} chunks @ 512 tokens each")

# ── 4. ProLAD Schedule ────────────────────────────────────────────
all_loras = get_loras(model)
N = len(all_loras)
def prolad_activate(step, total):
    p = step / max(total - 1, 1)
    n_active = max(1, int(N * (1 - math.cos(math.pi * p / 2))))
    for i, (_, mod) in enumerate(all_loras):
        mod.activate(i < n_active)
    return n_active, N

# ── 5. Training ───────────────────────────────────────────────────
print(f"\nProLAD training: {len(chunks)} steps, cosine schedule...")
opt = torch.optim.AdamW([p for n, p in model.named_parameters() if "lora" in n], lr=2e-4)
model.train()
losses = []
for step, chunk in enumerate(chunks):
    prolad_activate(step, len(chunks))
    input_ids = chunk.unsqueeze(0).to(device)
    loss = model(input_ids, labels=input_ids).loss
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 30 == 0:
        n_act, _ = prolad_activate(step, len(chunks))
        print(f"  Step {step:3d}/{len(chunks)} | Loss: {loss.item():.4f} | Active: {n_act}/{N}")
print(f"  Done: {losses[0]:.4f} → {losses[-1]:.4f}")

# Save LoRA weights
prolad_state = {n: p.data.clone() for n, p in model.named_parameters() if "lora" in n}

# ── 6. Baseline Training (immediate) ──────────────────────────────
print("\nTraining baseline (immediate schedule)...")
for n, p in model.named_parameters():
    if "lora_A" in n: nn.init.normal_(p, 0, 0.02)
    elif "lora_B" in n: nn.init.zeros_(p)
all_loras = get_loras(model)
for _, mod in all_loras:
    mod.activate(True)

opt2 = torch.optim.AdamW([p for n, p in model.named_parameters() if "lora" in n], lr=2e-4)
model.train()
losses2 = []
for step, chunk in enumerate(chunks):
    input_ids = chunk.unsqueeze(0).to(device)
    loss = model(input_ids, labels=input_ids).loss
    opt2.zero_grad(); loss.backward(); opt2.step()
    losses2.append(loss.item())
    if step % 30 == 0:
        print(f"  Step {step:3d}/{len(chunks)} | Loss: {loss2[-1] if len(losses2) > 0 else 0:.4f}")
# Fix the print
print(f"  Done: {losses2[0]:.4f} → {losses2[-1]:.4f}")

baseline_state = {n: p.data.clone() for n, p in model.named_parameters() if "lora" in n}

# ── 7. Benchmark — PPL + Kod Kalitesi ────────────────────────────
print("\n" + "=" * 70)
print("BENCHMARK")
print("=" * 70)

test_prompts = [
    "### Instruction:\nWrite a Python function to check if a number is prime.\n\n### Response:\n",
    "### Instruction:\nWrite a binary search algorithm in Python.\n\n### Response:\n",
    "### Instruction:\nCreate a Python class for a stack data structure.\n\n### Response:\n",
    "### Instruction:\nWrite a function to reverse a linked list.\n\n### Response:\n",
    "### Instruction:\nWrite a function that returns the sum of two numbers.\n\n### Response:\n",
]

def eval_state(state_dict, name):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in state_dict:
                p.data.copy_(state_dict[n])

    results = []
    for prompt in test_prompts:
        enc = tokenizer(prompt, return_tensors="pt").to(device)

        # Base only (LoRA off)
        set_lora(model, False)
        t0 = time.perf_counter()
        out_base = model.generate(**enc, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        base_ms = (time.perf_counter() - t0) * 1000
        base_text = tokenizer.decode(out_base[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

        # PPL base
        with torch.no_grad():
            ppl_base = math.exp(F.cross_entropy(model(enc['input_ids']).logits[0, :-1], enc['input_ids'][0, 1:]).item())

        # LoRA on
        set_lora(model, True)
        t0 = time.perf_counter()
        out_lora = model.generate(**enc, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        lora_ms = (time.perf_counter() - t0) * 1000
        lora_text = tokenizer.decode(out_lora[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

        # PPL LoRA
        with torch.no_grad():
            ppl_lora = math.exp(F.cross_entropy(model(enc['input_ids']).logits[0, :-1], enc['input_ids'][0, 1:]).item())

        task = prompt.replace("### Instruction:\nWrite", "").split(".")[0] if "Write" in prompt else prompt[:50]
        results.append({
            "state": name,
            "task": task.strip(),
            "ppl_base": round(ppl_base, 2),
            "ppl_lora": round(ppl_lora, 2),
            "base_ms": round(base_ms, 1),
            "lora_ms": round(lora_ms, 1),
            "speedup": round(lora_ms / max(base_ms, 0.01), 2),
            "base_output": base_text[:200],
            "lora_output": lora_text[:200],
        })

        print(f"\n{'─'*60}")
        print(f"[{name}] Task: {task}")
        print(f"  PPL: base={ppl_base:.2f}  LoRA={ppl_lora:.2f}  Gap={ppl_lora-ppl_base:.2f}")
        print(f"  Latency: base={base_ms:.0f}ms  LoRA={lora_ms:.0f}ms  Speedup={lora_ms/max(base_ms,0.01):.2f}x")
        if "ProLAD" in name:
            print(f"  Base:  {base_text[:150]}")
            print(f"  LoRA:  {lora_text[:150]}")

    return results

print("\nEvaluating ProLAD...")
prolad_r = eval_state(prolad_state, "ProLAD")

print("\nEvaluating Baseline...")
baseline_r = eval_state(baseline_state, "Baseline")

# ── 8. Summary ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"\n{'Method':<15} {'Task':<30} {'PPL(base)':<12} {'PPL(LoRA)':<12} {'Speedup':<10}")
print(f"{'─'*15} {'─'*30} {'─'*12} {'─'*12} {'─'*10}")

all_results = prolad_r + baseline_r
for r in all_results:
    print(f"{r['state']:<15} {r['task'][:28]:<30} {r['ppl_base']:<12} {r['ppl_lora']:<12} {r['speedup']:<10}")

# DataFrame
df = pd.DataFrame(all_results)
df.to_csv("/kaggle/working/coding_benchmark.csv", index=False)

summary = {
    "model": MODEL_ID,
    "train_samples": len(texts),
    "train_steps": len(chunks),
    "prolad": prolad_r,
    "baseline": baseline_r,
}
with open("/kaggle/working/coding_results.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n✅ Saved: coding_benchmark.csv + coding_results.json")
print("Done!")
